# Kadry Mostafa - MiniLM Semantic Search

This notebook covers milestone 3: add semantic search using MiniLM embeddings, compare it with BM25, and log nDCG@10.

## 1. Setup

The notebook reuses the project modules so the notebook, scripts, tests, and reports all run the same retrieval logic.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'stacklite_questions.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bm25_retriever import BM25Retriever, load_stacklite_dataset
from src.evaluation import evaluate_retrieval, load_evaluation_questions
from src.semantic_retriever import DEFAULT_MINILM_MODEL, SemanticRetriever

DATASET_PATH = PROJECT_ROOT / 'data' / 'stacklite_questions.csv'
QUESTIONS_PATH = PROJECT_ROOT / 'evaluation' / 'bm25_eval_queries.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'MiniLM model: {DEFAULT_MINILM_MODEL}')

Project root: C:\Users\RTX\Downloads\Advanced DL
MiniLM model: sentence-transformers/all-MiniLM-L6-v2


## 2. Load Corpus And Evaluation Questions

In [2]:
questions = load_evaluation_questions(QUESTIONS_PATH)
documents = load_stacklite_dataset(DATASET_PATH)

print(f'Loaded documents: {len(documents)}')
print(f'Evaluation questions: {len(questions)}')
for question in questions:
    print(f"- {question.query_id}: {question.query}")

Loaded documents: 1500
Evaluation questions: 5
- q1_micro_macro_average: Why are micro average precision recall and F1 equal in multiclass classification?
- q2_ai_vs_ml: What is the difference between artificial intelligence and machine learning?
- q3_deconvolution_layers: What are deconvolutional layers in convolutional neural networks?
- q4_keras_class_weights: How do I set class weights for imbalanced classes in Keras?
- q5_dying_relu: What is the dying ReLU problem in neural networks?


## 3. Build BM25 And MiniLM Retrievers

In [3]:
bm25_retriever = BM25Retriever(k1=1.5, b=0.75).fit(documents)
semantic_retriever = SemanticRetriever(model_name=DEFAULT_MINILM_MODEL).fit(documents)

print('BM25 index ready')
print('MiniLM semantic index ready')

BM25 index ready
MiniLM semantic index ready


## 4. Evaluate And Compare

In [4]:
def make_run(retriever):
    return {
        question.query_id: [
            str(result['doc_id']) for result in retriever.search(question.query, top_k=10)
        ]
        for question in questions
    }

bm25_run = make_run(bm25_retriever)
semantic_run = make_run(semantic_retriever)

bm25_metrics, bm25_rows = evaluate_retrieval(bm25_run, questions, k=10)
semantic_metrics, semantic_rows = evaluate_retrieval(semantic_run, questions, k=10)

print('BM25 metrics')
print(f"MAP@10: {bm25_metrics['MAP@10']:.6f}")
print(f"MRR@10: {bm25_metrics['MRR@10']:.6f}")
print(f"nDCG@10: {bm25_metrics['nDCG@10']:.6f}")
print()
print('MiniLM semantic metrics')
print(f"MAP@10: {semantic_metrics['MAP@10']:.6f}")
print(f"MRR@10: {semantic_metrics['MRR@10']:.6f}")
print(f"nDCG@10: {semantic_metrics['nDCG@10']:.6f}")

BM25 metrics
MAP@10: 1.000000
MRR@10: 1.000000
nDCG@10: 1.000000

MiniLM semantic metrics
MAP@10: 1.000000
MRR@10: 1.000000
nDCG@10: 1.000000


## 5. Top Semantic Hits

In [5]:
for question in questions:
    results = semantic_retriever.search(question.query, top_k=3)
    print(f"\nQuery: {question.query}")
    for result in results:
        print(
            f"  #{result['rank']} {result['doc_id']} | "
            f"score={result['score']:.6f} | {result['title']}"
        )


Query: Why are micro average precision recall and F1 equal in multiclass classification?
  #1 datascience:15989 | score=0.739459 | Micro Average vs Macro average Performance in a Multiclass classification setting
  #2 datascience:36817 | score=0.628729 | Why is the F-measure preferred for classification tasks?
  #3 datascience:40900 | score=0.609956 | What's the difference between Sklearn F1 score 'micro' and 'weighted' for a multi class classification problem?

Query: What is the difference between artificial intelligence and machine learning?
  #1 ai:35 | score=0.885910 | What is the difference between artificial intelligence and machine learning?
  #2 datascience:19077 | score=0.869088 | Difference between machine learning and artificial intelligence
  #3 ai:7446 | score=0.703852 | What is the difference between artificial intelligence and computational intelligence?

Query: What are deconvolutional layers in convolutional neural networks?
  #1 datascience:6107 | score=0.704796 | W

## 6. Persist Script Outputs

The project script writes the CSV and Markdown report used for submission.

In [6]:
from scripts.run_semantic_evaluation import main as run_semantic_evaluation

run_semantic_evaluation()

MiniLM semantic search evaluation
Model: sentence-transformers/all-MiniLM-L6-v2
Queries: 5
BM25 nDCG@10: 1.000000
Semantic nDCG@10: 1.000000
BM25 MAP@10: 1.000000
Semantic MAP@10: 1.000000
Comparison CSV: C:\Users\RTX\Downloads\Advanced DL\results\semantic_vs_bm25_comparison.csv
Semantic top-10 run: C:\Users\RTX\Downloads\Advanced DL\results\semantic_evaluation_top10.csv
Report: C:\Users\RTX\Downloads\Advanced DL\reports\semantic_search_report.md
